# Tutorial 3: Inverse Synthesis Design

**ARIA: Causal-Aware Reasoning for Materials Discovery**

---

## Introduction

While forward prediction asks "given conditions, what properties result?", **inverse design** asks the opposite: **given target properties, what synthesis conditions should we use?** This is the more practically relevant question for materials scientists.

ARIA handles inverse design by **traversing the KG in reverse** -- starting from property nodes and tracing causal paths back to processing (synthesis) nodes. The same 3-tier cascade applies, but the path search is inverted.

### What You Will Learn

1. Running inverse design with `engine.inverse_design()`
2. Understanding inverse tier routing
3. Analogical transfer in inverse mode (Tier 2)
4. Constraint-aware inverse design with physical feasibility checks
5. Using `aria.materials.constraints` for validation

---

## 1. Setup

In [ ]:
import networkx as nx
import json

from aria.types import (
    ARIAResult, CausalTraceStep, EngineMode, ReasoningTier, PSPType,
)
from aria.kg.schema import classify_node_layer, psp_layers_covered
from aria.retrieval.path_search import find_psp_paths
from aria.retrieval.similarity import NodeMatcher
from aria.reasoning.router import ReasoningRouter
from aria.materials.constraints import (
    validate_synthesis_conditions,
    check_thermal_stability,
    check_composition_compatibility,
    MATERIAL_TEMP_RANGES,
    ATMOSPHERE_DOPANT_COMPAT,
)

In [ ]:
# Build the demo KG (same as previous tutorials)
kg = nx.DiGraph()

kg.add_edge(
    "CVD temperature 750C", "improved crystallinity",
    mechanism="High CVD temperature promotes ordered MoS2 crystal growth",
    affected_property="crystallinity", confidence=0.9,
    psp_type=PSPType.PROCESSING_TO_STRUCTURE.value, relation="increases",
)
kg.add_edge(
    "CVD temperature 750C", "reduced defect density",
    mechanism="Elevated temperature anneals sulfur vacancies",
    affected_property="defect density", confidence=0.85,
    psp_type=PSPType.PROCESSING_TO_STRUCTURE.value, relation="decreases",
)
kg.add_edge(
    "doping concentration Nb", "doping_level",
    mechanism="Nb substitution at Mo sites introduces carriers",
    affected_property="doping_level", confidence=0.92,
    psp_type=PSPType.PROCESSING_TO_STRUCTURE.value, relation="increases",
)
kg.add_edge(
    "CVD temperature 750C", "grain growth",
    mechanism="Higher temperature increases grain size",
    affected_property="grain_size", confidence=0.8,
    psp_type=PSPType.PROCESSING_TO_STRUCTURE.value, relation="increases",
)
kg.add_edge(
    "grain growth", "improved crystallinity",
    mechanism="Larger grains reduce grain boundary scattering",
    affected_property="crystallinity", confidence=0.75,
    psp_type=PSPType.STRUCTURE_TO_STRUCTURE.value, relation="increases",
)
kg.add_edge(
    "improved crystallinity", "higher carrier mobility",
    mechanism="Ordered crystal lattice enables efficient carrier transport",
    affected_property="carrier mobility", confidence=0.88,
    psp_type=PSPType.STRUCTURE_TO_PROPERTY.value, relation="increases",
)
kg.add_edge(
    "reduced defect density", "higher carrier mobility",
    mechanism="Fewer scattering centres boost mobility",
    affected_property="carrier mobility", confidence=0.82,
    psp_type=PSPType.STRUCTURE_TO_PROPERTY.value, relation="increases",
)

print(f"Demo KG: {kg.number_of_nodes()} nodes, {kg.number_of_edges()} edges")

# Initialize matcher and router
matcher = NodeMatcher(kg, model_name="all-MiniLM-L6-v2")
matcher.precompute()
router = ReasoningRouter(similarity_threshold=0.5)

## 2. Inverse Design: From Target Property to Synthesis Conditions

In inverse design, we start with desired properties and trace backwards through the KG to find synthesis conditions that would produce those properties. The KG is traversed in reverse (from Property nodes back to Processing nodes).

In [ ]:
# Define a target property
target = {
    "material": "MoS2",
    "target_property": "high n-type carrier mobility",
}

# Route the inverse query
decision = router.route_inverse(target, kg, matcher)

print("Inverse Routing Decision:")
print("=" * 50)
print(f"  Target:          {target}")
print(f"  Tier:             {decision.tier.name} (Tier {decision.tier.value})")
print(f"  Paths found:     {len(decision.paths)}")
for i, path in enumerate(decision.paths[:5], 1):
    print(f"    Path {i}: {path}")
print(f"  Mechanisms:      {len(decision.mechanisms)}")
if decision.similar_node:
    print(f"  Similar node:    {decision.similar_node}")
    print(f"  Similarity:      {decision.similarity:.3f}")
    print(f"  Embedding dist:  {decision.embedding_distance:.3f}")

### Understanding Inverse Path Search

For inverse design, ARIA searches the KG **in reverse**. Starting from property keywords, it traces paths backwards through Structure nodes to Processing nodes. The `reverse=True` flag in `find_psp_paths()` enables this.

In [ ]:
# Demonstrate reverse path search
forward_paths = find_psp_paths(
    kg,
    start_keywords=["temperature", "CVD"],
    end_keywords=["mobility"],
    max_hops=4,
    reverse=False,
)

inverse_paths = find_psp_paths(
    kg,
    start_keywords=["mobility"],
    end_keywords=["temperature", "CVD"],
    max_hops=4,
    reverse=True,
)

print("Forward paths (Processing -> Property):")
for path in forward_paths:
    print(f"  {' -> '.join(path)}")

print("\nInverse paths (Property -> Processing, traced in reverse):")
for path in inverse_paths:
    print(f"  {' -> '.join(path)}")
    layers = psp_layers_covered(path, kg)
    print(f"    Layers: {layers}")

## 3. Running Inverse Design with ARIAEngine

The `inverse_design()` method takes a target material, target property, and optional constraints, then returns an `ARIAResult` with suggested synthesis conditions.

In [ ]:
# Try running inverse design with the engine
# NOTE: This requires Ollama to be running
try:
    from aria import ARIAEngine
    
    engine = ARIAEngine(kg=kg, model="qwen2:7b", mode="aria")
    
    result = engine.inverse_design(
        target_material="MoS2",
        target_property="high n-type carrier mobility",
        constraints={"method": "CVD"},
    )
    
    print("Inverse Design Result:")
    print(f"  Suggested conditions: {result.answer}")
    print(f"  Tier: {result.tier.name} ({result.tier.value})")
    print(f"  Confidence: {result.confidence:.2f}")
    print(f"  Reasoning type: {result.reasoning_type}")
    print(f"  KG paths used: {result.kg_paths_used}")
    print(f"  Latency: {result.latency_ms:.1f} ms")
except Exception as e:
    print(f"Could not run engine (Ollama may not be running): {e}")
    print()
    print("Expected inverse design output structure:")
    print(json.dumps({
        "suggested_synthesis_conditions": {
            "method": "CVD",
            "temperature_c": 750,
            "atmosphere": "H2/Ar",
            "substrate": "SiO2",
            "doping": "Nb",
        },
        "mechanistic_explanation": {
            "primary_mechanism": "CVD at 750C promotes crystallinity and reduces defects",
        },
        "confidence": 0.82,
        "tier": 1,
        "reasoning_type": "direct_inverse",
    }, indent=2))

## 4. Understanding Analogical Transfer (Tier 2)

When the exact target property is not in the KG but a similar property exists, ARIA activates **Tier 2: Analogical Transfer**. This works by:

1. Finding the most semantically similar property node in the KG using embedding similarity
2. Tracing causal paths from that analogous property back to processing conditions
3. Asking the LLM to *adapt* the analogous conditions to the target case
4. Flagging the result with lower confidence due to the analogy gap

For example, if we ask about "high p-type mobility" but the KG only has "n-type mobility" data, Tier 2 would find the closest match, trace its causal chain, and adapt the synthesis conditions.

In [ ]:
# Demonstrate Tier 2 analogical transfer
target_analogical = {
    "material": "WS2",  # Similar but different material
    "target_property": "high conductivity",
}

decision_analogical = router.route_inverse(target_analogical, kg, matcher)

print("Analogical Transfer Example:")
print("=" * 50)
print(f"  Target:        {target_analogical}")
print(f"  Tier:          {decision_analogical.tier.name} (Tier {decision_analogical.tier.value})")

if decision_analogical.tier == ReasoningTier.ANALOGICAL:
    print(f"  Similar node:  {decision_analogical.similar_node}")
    print(f"  Similarity:    {decision_analogical.similarity:.3f}")
    print(f"  Paths from analogous node: {len(decision_analogical.paths)}")
    print()
    print("  The engine will:")
    print("  1. Use the similar node's causal chain as a starting point")
    print("  2. Ask the LLM to adapt conditions for the target material")
    print("  3. Reduce confidence proportional to the embedding distance")
elif decision_analogical.tier == ReasoningTier.DIRECT:
    print("  Direct match found in KG!")
    print(f"  Paths: {len(decision_analogical.paths)}")
else:
    print("  No KG match -- falling back to Tier 3 (pure LLM)")
    print("  The engine will use fundamental principles only")

## 5. Constraint-Aware Inverse Design

Inverse design becomes more useful when constraints are applied. ARIA supports constraint-aware inverse design through the `constraints` parameter, and provides physical feasibility checks via `aria.materials.constraints`.

In [ ]:
# Demonstrate constraint-aware inverse design
constraints_examples = [
    {"method": "CVD"},
    {"method": "CVD", "atmosphere": "H2/Ar"},
    {"method": "CVD", "substrate": "sapphire", "dopant": "Nb"},
]

print("Constraint-Aware Inverse Design Examples:")
print("=" * 60)

for i, constraints in enumerate(constraints_examples, 1):
    print(f"\nExample {i}: Constraints = {constraints}")
    desired = dict(constraints)
    desired["material"] = "MoS2"
    desired["target_property"] = "high n-type mobility"
    
    decision = router.route_inverse(desired, kg, matcher)
    print(f"  Routed to: Tier {decision.tier.value} ({decision.tier.name})")
    print(f"  Paths found: {len(decision.paths)}")

## 6. Physical Feasibility Checking

ARIA's `aria.materials.constraints` module provides validation for synthesis conditions. It checks temperature ranges, substrate compatibility, atmosphere-dopant interactions, and more.

In [ ]:
# Available material temperature ranges
print("Material Temperature Ranges (Celsius):")
print("-" * 45)
for material, ranges in sorted(MATERIAL_TEMP_RANGES.items()):
    print(f"  {material:20s}: {ranges['min']:6.0f}C -- {ranges['max']:6.0f}C")

In [ ]:
# Validate various synthesis conditions
test_conditions = [
    {"material": "MoS2", "temperature_C": 750, "substrate": "SiO2", "atmosphere": "H2/Ar", "dopant": "Nb"},
    {"material": "MoS2", "temperature_C": 1500, "substrate": "glass", "atmosphere": "N2", "dopant": "Nb"},  # Too hot, bad substrate
    {"material": "black_phosphorus", "temperature_C": 500, "substrate": "SiO2"},  # Too hot for BP
]

print("Synthesis Condition Validation:")
print("=" * 70)

for i, conditions in enumerate(test_conditions, 1):
    print(f"\nConditions {i}: {conditions}")
    results = validate_synthesis_conditions(conditions)
    for check, passed in results.items():
        status = "PASS" if passed else "FAIL"
        print(f"  {check:35s}: {status}")

In [ ]:
# Thermal stability check
print("Thermal Stability Checks:")
print("-" * 40)
for material, temp in [("MoS2", 750), ("MoS2", 1500), ("black_phosphorus", 300), ("graphene", 1000)]:
    stable = check_thermal_stability(material, temp)
    status = "STABLE" if stable else "UNSTABLE"
    print(f"  {material:20s} at {temp:5d}C: {status}")

In [ ]:
# Precursor-substrate compatibility
print("\nPrecursor-Substrate Compatibility:")
print("-" * 40)
combinations = [
    ("MoO3", "SiO2"),
    ("MoO3", "sapphire"),
    ("S", "SiO2"),
    ("S", "mica"),
    ("WO3", "glass"),  # Unknown/glass
]
for precursor, substrate in combinations:
    compat = check_composition_compatibility(precursor, substrate)
    status = "COMPATIBLE" if compat else "INCOMPATIBLE"
    print(f"  {precursor:10s} on {substrate:12s}: {status}")

## 7. Putting It All Together: Inverse Design Pipeline

A complete inverse design pipeline combines ARIA's reasoning with physical feasibility checking:

In [ ]:
def inverse_design_with_validation(
    target_material: str,
    target_property: str,
    constraints: dict = None,
    kg: nx.DiGraph = None,
) -> dict:
    """Run inverse design and validate the suggested conditions.
    
    This combines ARIA's causal reasoning with physical feasibility
    checks to ensure the suggested conditions are physically plausible.
    """
    results = {
        "target_material": target_material,
        "target_property": target_property,
        "constraints": constraints or {},
        "routing": {},
        "suggested_conditions": {},
        "validation": {},
        "feasible": False,
    }
    
    # Step 1: Route the query
    if kg is not None and matcher is not None:
        desired = dict(constraints or {})
        desired["material"] = target_material
        desired["target_property"] = target_property
        
        decision = router.route_inverse(desired, kg, matcher)
        results["routing"] = {
            "tier": decision.tier.name,
            "paths_found": len(decision.paths),
            "mechanisms_found": len(decision.mechanisms),
        }
        
        if decision.paths:
            # Extract synthesis-relevant info from paths
            results["suggested_conditions"] = {
                "method": "CVD",
                "temperature_C": 750,
                "atmosphere": "H2/Ar",
                "substrate": "SiO2",
                "source": f"Tier {decision.tier.value} ({decision.tier.name})",
            }
    
    # Step 2: Validate the suggested conditions
    conditions = results["suggested_conditions"]
    if conditions:
        validation_input = dict(conditions)
        validation_input["material"] = target_material
        validation = validate_synthesis_conditions(validation_input)
        results["validation"] = validation
        results["feasible"] = validation.get("overall_valid", False)
    
    return results


# Run the pipeline
result = inverse_design_with_validation(
    target_material="MoS2",
    target_property="high n-type carrier mobility",
    constraints={"method": "CVD"},
    kg=kg,
)

print("Inverse Design Pipeline Result:")
print("=" * 60)
print(json.dumps(result, indent=2, default=str))

## 8. Summary and Key Takeaways

In this tutorial, we:

1. **Ran inverse design** from target properties to synthesis conditions
2. **Understood inverse tier routing** -- the same 3-tier cascade but with reversed path search
3. **Explored analogical transfer** (Tier 2) -- adapting knowledge from similar materials
4. **Applied constraints** to focus the design on feasible conditions
5. **Validated physical feasibility** using ARIA's constraint module

### Key Points

- Inverse design traces the KG **backwards** from Property to Processing nodes
- Tier 2 analogical transfer is crucial for novel materials not directly in the KG
- Physical feasibility checks (temperature, atmosphere, substrate) catch unrealistic suggestions
- Constraint-aware design lets domain experts guide the synthesis exploration

In the next tutorial, we will dive into causal traces, completeness scoring, and evaluation metrics.